Ingesting the data

In [1]:
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
df = pd.read_csv('/content/drive/MyDrive/ab_data.csv')

In [4]:
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


EDA -- Missing values

In [5]:
df.isna().sum()

,0
user_id,0
timestamp,0
group,0
landing_page,0
converted,0


In [6]:
df["user_id"].duplicated().sum()

np.int64(3894)

EDA - duplicated values

In [7]:
mismatch = ((df.group == "treatment") != (df.landing_page == "new_page"))
print(mismatch.sum())
print(df.user_id.duplicated().sum())

df = df[~mismatch]
df = df[~df.user_id.duplicated(keep = 'first')]
print(df.user_id.duplicated().sum())

print(f"Number of users: {df.user_id.count()}")

3893
3894
0
Number of users: 290584


Found 3,893 rows where a user’s group didn’t match the page they saw, and 1 user who appeared twice. Removed both, leaving 290,584 users.

Conversion rates: What percentage of users converted in each group?

In [8]:
df.columns

Index(['user_id', 'timestamp', 'group', 'landing_page', 'converted'], dtype='object')

In [9]:
rates = df.groupby("group")["converted"].mean()
rates

,converted
group,
control,0.120386
treatment,0.118808


In [10]:
control = rates["control"]
treatment = rates["treatment"]
difference = treatment-control
print(f"Difference conversion rate: {difference*100:.2f}")

Difference conversion rate: -0.16


Effect size: Calculate the absolute difference in conversion rates and relative lift.

In [11]:
relative_lift = (treatment-control)/control
print(f"Relative lift: {relative_lift*100:.2f}")


Relative lift: -1.31


Step 5: Define the hypotheses

Hypotheses — A/B Test

Null Hypothesis (H₀): There is no difference in conversion rates between the old landing page (control) and the new landing page (treatment).

\\[ H_0:p\_{\text{treatment}}=p\_{\text{control}} \\]

Alternative Hypothesis (H₁): There is a difference in conversion rates between the old landing page and the new landing page.

\\[ H_1:p\_{\text{treatment}}\neq p\_{\text{control}} \\]

Significance level: α = 0.05 (5%)

Test: Two-sided two-proportion z-test.



In [12]:
summary = df.groupby('group')['converted'].agg(['sum', 'count'])
print(summary)

             sum   count
group                   
control    17489  145274
treatment  17264  145310


Sample ratio check: the groups are nearly identical in size (145,274 control vs. 145,310 treatment, about 50/50), so the random split worked as expected.

In [13]:
from statsmodels.stats.proportion import proportions_ztest

conversions = [
    summary.loc['treatment', 'sum'],
    summary.loc['control', 'sum']
]

sample_sizes = [
    summary.loc['treatment', 'count'],
    summary.loc['control', 'count']
]

z_stat, p_value = proportions_ztest(
    count=conversions,
    nobs=sample_sizes,
    alternative='two-sided'
)

print("Z-statistic:", z_stat)
print("P-value:", p_value)

Z-statistic: -1.3109241984234394
P-value: 0.18988337448195103


In [14]:
from statsmodels.stats.proportion import confint_proportions_2indep

lower, upper = confint_proportions_2indep(
    count1=conversions[0],
    nobs1=sample_sizes[0],
    count2=conversions[1],
    nobs2=sample_sizes[1],
    compare='diff',
    method='newcomb',
    alpha=0.05
)

print(f"95% CI: [{lower * 100:.2f}, {upper * 100:.2f}] percentage points")

95% CI: [-0.39, 0.08] percentage points


# **Conclusion**

The new page converted at 11.88% versus 12.04% for the old page, a difference of -0.16 percentage points (p = 0.19; 95% CI -0.39 to 0.08). The difference isn’t statistically significant, and the interval rules out any meaningful improvement. Recommendation: don’t launch the new page.